# 07 — Verdict

Applies the frozen §8 / §9 / §10 decision rules via `src.criteria.evaluate`. The verdict
is computed **from the rules as written**, not from reading the numbers — that is the
whole reason the criteria are code.

This notebook performs no new statistics. It consumes notebook 05 and 06 outputs.

## 1 — Header

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ -- tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
from src.utils import log_versions, run_timestamp, record_provenance, write_report, write_sha256_manifest

RUN_TIMESTAMP = run_timestamp()
VERSIONS = log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'matplotlib'])
print(f'Run: {RUN_TIMESTAMP}')
assert sys.version_info[:2] == (3, 12), f'Expected Python 3.12, got {sys.version_info[:2]}'
print(f'Python: {sys.version.split()[0]}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-bir-dependency


Library versions:
  pandas: 2.3.3
  numpy: 2.5.4
  scipy: 1.18.1
  sklearn: 1.9.1
  matplotlib: 3.11.2
Run: 2026-10-12T01:00:01Z
Python: 3.12.13


## 2 — Assemble the frozen inputs

In [2]:
import json as _json
from src.utils import PROCESSED_DIR
from src.criteria import evaluate, ALPHA, SUPPORT_DELTA, INCONCLUSIVE_MIN_ABS_DELTA, FALSIFIED_MAX_ABS_DELTA

primary_payload = _json.loads((PROCESSED_DIR / 'primary_result.json').read_text())
extras = _json.loads((PROCESSED_DIR / 'robustness_extras.json').read_text())
per_gene_all = pd.read_csv(PROCESSED_DIR / 'per_gene_results.csv')
rob = pd.read_csv(PROCESSED_DIR / 'robustness_table.csv')
loo = pd.read_csv(PROCESSED_DIR / 'loo_deltas.csv')

primary = primary_payload['contrasts']['primary_median_8v8']
tertile = primary_payload['contrasts']['sensitivity_tertile_3v13']
per_gene = per_gene_all[per_gene_all['contrast'] == 'primary_median_8v8'].reset_index(drop=True)
r1 = rob[rob['check'] == 'R1'].iloc[0].to_dict()
r2 = rob[rob['check'] == 'R2'].iloc[0].to_dict()
interaction = extras['interaction_6_2']

print(f"primary   : delta={primary['cliffs_delta']:+.4f}  p={primary['p_value']:.4f}")
print(f"tertile   : delta={tertile['cliffs_delta']:+.4f}  p={tertile['p_value']:.4f}")
print(f"R1 gold   : delta={r1['cliffs_delta']:+.4f}  p={r1['p_value']:.4f}")
print(f"R7 refits : {len(loo)}  range [{loo['delta'].min():+.4f}, {loo['delta'].max():+.4f}]")
print(f"min q     : {per_gene['q_value'].min():.4f}")

primary   : delta=+0.2812  p=0.3823
tertile   : delta=+0.5385  p=0.1893
R1 gold   : delta=-0.6190  p=0.1833
R7 refits : 16  range [+0.1786, +0.4643]
min q     : 0.9591


## 3 — Render the verdict

In [3]:
verdict = evaluate(
    primary=primary,
    per_gene=per_gene,
    r1_gold_only=r1,
    r2_median=r2,
    loo_deltas=loo['delta'].tolist(),
    interaction=interaction,
    tertile=tertile,
)
print('\n'.join(verdict.as_markdown()))

**Outcome: `indeterminate`** (none of Sections 8/9/10)

The result matches no pre-specified outcome bucket: p=0.3823, delta=+0.281. Failed support conditions: ['primary', 'bir_genes', 'r7_loo']. Report this as-is; do not round it toward the nearest verdict.

| Condition | Met | Detail |
|---|---|---|
| S8.1 primary p<0.05 and delta <= -0.33 | NO | p=0.3823, delta=+0.281 |
| S8.2 gold-only same direction, |delta| >= 0.30 | yes | gold-only delta=-0.619 |
| S8.3 >=2/3 BIR genes at q<0.10, no HR gene stronger | NO | 0/3 BIR genes at q<0.1 in predicted direction (none); strongest BIR delta -0.344; strongest same-direction HR delta -0.250 |
| S8.4 no sign flip on leave-one-line-out | NO | 16 refits, delta range [+0.179, +0.464] |
| S8.5 ALT- stratum shows no significant same-direction shift | yes | ALT- delta=-0.400, p=0.4127; ALT+ delta=+0.281 (delta difference reported as a diagnostic, not a gate) |

**Section 6.4 concentration comparison** (descriptive, non-gating):

- delta, 8v8 median p

## 4 — Why not the other buckets

Shown explicitly so the reader can check the arithmetic rather than take the label on
trust.

In [4]:
pv, d = primary['p_value'], primary['cliffs_delta']
checks = [
    ('§8  supported',    f'p<{ALPHA} and delta<={SUPPORT_DELTA}',          pv < ALPHA and d <= SUPPORT_DELTA),
    ('§9  inconclusive', f'p>={ALPHA} and delta<=-{INCONCLUSIVE_MIN_ABS_DELTA}', pv >= ALPHA and d <= -INCONCLUSIVE_MIN_ABS_DELTA),
    ('§10 clean null',   f'p>={ALPHA} and |delta|<{FALSIFIED_MAX_ABS_DELTA}',    pv >= ALPHA and abs(d) < FALSIFIED_MAX_ABS_DELTA),
    ('§10 wrong-dir sig',f'p<{ALPHA} and delta>0',                        pv < ALPHA and d > 0),
]
for label, rule, met in checks:
    print(f'  {label:<20s} {rule:<34s} -> {met}')
print(f'\nobserved: p = {pv:.4f}, delta = {d:+.4f}, |delta| = {abs(d):.4f}')
print(f'\n=> {verdict.outcome.upper()}: matches no frozen bucket.')
print(f'   |delta| = {abs(d):.3f} exceeds the {FALSIFIED_MAX_ABS_DELTA} clean-null ceiling, but the sign is')
print(f'   OPPOSITE to the prediction and p is far from {ALPHA}. §11 is therefore NOT authorized.')

  §8  supported        p<0.05 and delta<=-0.33            -> False
  §9  inconclusive     p>=0.05 and delta<=-0.4            -> False
  §10 clean null       p>=0.05 and |delta|<0.2            -> False
  §10 wrong-dir sig    p<0.05 and delta>0                 -> False

observed: p = 0.3823, delta = +0.2812, |delta| = 0.2812

=> INDETERMINATE: matches no frozen bucket.
   |delta| = 0.281 exceeds the 0.2 clean-null ceiling, but the sign is
   OPPOSITE to the prediction and p is far from 0.05. §11 is therefore NOT authorized.


## 5 — Direction of every pre-specified check

In [5]:
summary = rob[['check', 'description', 'n_low', 'n_intact', 'cliffs_delta',
               'p_value', 'direction_as_predicted']].copy()
summary['n'] = summary['n_low'].astype(str) + 'v' + summary['n_intact'].astype(str)
print(summary[['check', 'n', 'cliffs_delta', 'p_value', 'direction_as_predicted']]
      .round(4).to_string(index=False))
print()
r4 = extras['R4_continuous']
print(f"R4 continuous: rho = {r4['rho']:+.4f}  p = {r4['p_value']:.4f}  "
      f"predicted rho>0: {r4['as_predicted']}")
n_pred = int(summary['direction_as_predicted'].sum())
print(f'\nChecks in the predicted direction: {n_pred} of {len(summary)}')
smallest = summary.nsmallest(2, 'n_low')[['check', 'n']].to_dict('records')
print(f'The checks favouring the hypothesis are the two smallest comparisons: {smallest}')

check    n  cliffs_delta  p_value  direction_as_predicted
   R1  3v7       -0.6190   0.1833                    True
   R2  8v8        0.2812   0.3823                   False
   R3 5v11        0.5636   0.0897                   False
   R5  7v8        0.1786   0.6126                   False
   R6  4v5       -0.4000   0.4127                    True
   R8  8v8        0.1562   0.6454                   False
  6.3  8v8        0.2812   0.3823                   False

R4 continuous: rho = -0.4415  p = 0.0869  predicted rho>0: False

Checks in the predicted direction: 2 of 7
The checks favouring the hypothesis are the two smallest comparisons: [{'check': 'R1', 'n': '3v7'}, {'check': 'R6', 'n': '4v5'}]


## 6 — Report

In [6]:
verdict_path = PROCESSED_DIR / 'stage1_verdict.json'
verdict_path.write_text(_json.dumps(verdict.as_dict(), indent=2, sort_keys=True) + '\n')

lines = [
    '# Notebook 07 — Verdict',
    '',
    'Computed by `src.criteria.evaluate` from the frozen §8/§9/§10 rules. No new',
    'statistics are performed here.',
    '',
    f'## Outcome: `{verdict.outcome}`',
    '',
    f'**Section:** {verdict.section}',
    '',
    verdict.headline,
    '',
    '## Plain-language reading',
    '',
    '**The hypothesis was not supported.** The primary point estimate ran *opposite* to',
    'the prediction and nothing approached significance:',
    '',
    f"- primary (8v8): Cliff's delta **{primary['cliffs_delta']:+.4f}** (predicted < 0), "
    f"p = **{primary['p_value']:.4f}**",
    f"- pre-specified tertile sensitivity (3v13): delta **{tertile['cliffs_delta']:+.4f}**, "
    f"p = **{tertile['p_value']:.4f}**",
    f"- minimum BH q across the six primary genes: **{per_gene['q_value'].min():.4f}**",
    f"- R7: no refit of 16 reaches the predicted direction; range "
    f"**[{loo['delta'].min():+.4f}, {loo['delta'].max():+.4f}]**",
    f"- R4 continuous: Spearman rho **{extras['R4_continuous']['rho']:+.4f}** "
    f"(predicted > 0), p = {extras['R4_continuous']['p_value']:.4f}",
    '',
    'The two checks that do favour the hypothesis (R1 gold-only, delta −0.619 at 3v7; and',
    'R6, the ALT− control, delta −0.400 at 4v5) are the two **smallest** comparisons, where',
    'the exact null median |delta| is 0.30–0.33 and P(|delta| ≥ 0.474) is 27–29%. R6 is',
    'moreover the negative control, where the hypothesis predicts *no* effect, so a',
    '"predicted-direction" result there counts against a TCAB1-specific account, not for it.',
    '',
    '## Verdict conditions',
    '',
] + verdict.as_markdown()[4:] + [
    '',
    '## Why not the other buckets',
    '',
    '| Bucket | Rule | Met |',
    '|---|---|---|',
]
for label, rule, met in checks:
    lines.append(f'| {label} | `{rule}` | {met} |')
lines += [
    '',
    f"Observed: p = {pv:.4f}, delta = {d:+.4f}, |delta| = {abs(d):.4f}.",
    '',
    f'`indeterminate` is the honest label: |delta| = {abs(d):.3f} exceeds the',
    f'{FALSIFIED_MAX_ABS_DELTA} clean-null ceiling frozen in §10, yet the sign is opposite to',
    f'the prediction and p is far from {ALPHA}. **§11 (negative-result fallback) is therefore',
    'NOT authorized** — §10 did not trigger.',
    '',
    '## Methods limitation found by this result',
    '',
    "§10's clean-null ceiling (|delta| < 0.20) was calibrated for a null *in the predicted",
    'direction*. It did not anticipate a non-significant estimate of moderate size pointing',
    'the **wrong** way, which is what occurred. The rule is left exactly as frozen and this',
    'is recorded as a lesson for any v2 pre-specification. It was **not** retrofitted after',
    'seeing the result.',
    '',
    '## Every pre-specified check',
    '',
    '| Check | n | delta | p | Predicted direction? |',
    '|---|---|---|---|---|',
]
for _, r in summary.iterrows():
    lines.append(f"| {r['check']} | {r['n']} | {r['cliffs_delta']:+.4f} "
                 f"| {r['p_value']:.4f} | {'yes' if r['direction_as_predicted'] else 'no'} |")
lines += [
    f"| R4 | 16 | rho {r4['rho']:+.4f} | {r4['p_value']:.4f} "
    f"| {'yes' if r4['as_predicted'] else 'no'} |",
    '',
    f'In the predicted direction: **{n_pred} of {len(summary)}** checks.',
    '',
    '## Next step',
    '',
    'See `docs/stage1_outcome_memo.md`.',
]
report = write_report('07', lines)
manifest = write_sha256_manifest('07', [report, verdict_path])
record_provenance([{
    'notebook': '07', 'run_timestamp': RUN_TIMESTAMP,
    'outcome': verdict.outcome, 'section': verdict.section,
    'primary_delta': d, 'primary_p': pv,
}])
print(f'wrote {verdict_path.name}, {report.name}, {manifest.name}')
print(f'\nVERDICT: {verdict.outcome}')

wrote stage1_verdict.json, notebook_07_report.md, notebook_07_sha256.json

VERDICT: indeterminate
